# Glyph Agent

This notebook will serve as a **tutorial** on how to use the Glyph Agent class for glyph comparaison

## Installation & Setup

### Installation in windows

Using pip to install the necessary python packages

In [ ]:
%pip install torch
%pip install torchvision
%pip install pillow
%pip install numpy
%pip install scikit-learn
%pip install matplotlib
%pip install pandas

### Setup

The following drive [link](https://drive.google.com/drive/u/4/folders/17aQ-lMP1DIuvpGkbnDa7EnNU9P913vTF) has folders containing the dataset and the model that learned from it.

In each folder's name you can find the assigned difficulty which indicates the compexity of the data that the model learned from.

Please chose a model and its corresponding dataset and place them in this repository.

## Agent Tutorial

Let's start by defining our glyph

**Simple Star**

In [ ]:
import mglyph as mg
import numpy as np
from typing import Callable

def simple_star(x: float, canvas: mg.Canvas) -> None:
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    radius = mg.lerp(x, 0.01, canvas.ysize / 2)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * np.pi / 5, radius))
        vertices.append(mg.orbit(canvas.center, (segment + 0.5) * 2 * np.pi / 5,
                         np.cos(2 * np.pi / 5) / np.cos(np.pi / 5) * radius))

    canvas.polygon(vertices, linecap='round', style='fill', color='white') 
    canvas.polygon(vertices, width='70p', linecap='round', style='stroke', color='navy')

mg.show(simple_star)
mg.export(simple_star, path='data/simple-star.mglyph', short_name="star", 
          version="1.0.0", name="Simple Line", xvalues=np.linspace(0.0, 100.0, 10000))

**Simple Square**

In [ ]:
import mglyph as mg
import numpy as np
from typing import Callable


def simple_square(x: float, canvas: mg.Canvas) -> None:
    tl = (mg.lerp(x, 0.0, -1), mg.lerp(x, 0.0, -1.0))
    br = (mg.lerp(x, 0, 1), mg.lerp(x, 0, 1))
    canvas.rect(tl, br, color='red', style='stroke', width='60p')

mg.show(simple_square)
mg.export(simple_square, path='data/simple-square.mglyph', short_name="square", 
          version="1.0.0", name="Simple square", xvalues=np.linspace(0.0, 100.0, 10000))

**Simple Circle**

In [ ]:
import mglyph as mg
import numpy as np
from typing import Callable

def simple_circle(x: float, canvas: mg.Canvas) -> None:
    canvas.circle(canvas.center, mg.lerp(x, 0.01, canvas.ysize/2),
                  color='green', style='stroke', width='70p')

mg.show(simple_circle)
mg.export(simple_circle, path='data/simple-circle.mglyph', short_name="circle", 
          version="1.0.0", name="Simple Circle", xvalues=np.linspace(0.0, 100.0, 10000))

**Simple letter B**

In [ ]:
import mglyph as mg
import numpy as np
from typing import Callable

def b_letter() -> Callable[[float, mg.Canvas], None]:
    color = "orange"
    character = random.choice('B')
    return lambda x, canvas: scaled_letter(x, canvas, color, character)

mg.show(b_letter)
mg.export(b_letter, path='data/b-letter.mglyph', short_name="letter", 
          version="1.0.0", name="Simple B letter", xvalues=np.linspace(0.0, 100.0, 10000))

Let's start by importing the GlyphAgent class and defining our agent.

""
**glyph_filename**: path/folder name of the dataset

**model_filename**: path/file name of the model

**name**: name of the said agent. Can be ignored

**device**: chose which device to run on
""

In [ ]:
from src.glyph_agent import GlyphAgent

agent = GlyphAgent(
    glyph_data='data/simple-star.mglyph',
    model_filename='data/simple-star.pt',
    name="StarAI",
    device="cuda:0"
)

After the creation of the agent we can define the task.

""
**x1**: The value that you want to look for in the data folder of the first glyph

**x2**: The value that you want to look for in the data folder of the second glyph

**distance**: The distance from the specified values that the agent is permissible to cross. 
""

In [ ]:
task = {"x1": 42.9, "x2": 81.3, "distance": 1}

Now let's get the response of the agent for said task

**Verbose**: set to False by default. If changed to True then we would print the results

In [ ]:
response = agent.get_response({"x1": 41.9353, "x2": 81.3, "distance": 1}, verbose=True)

It is possible to define multiple tasks and get responses for

In [ ]:
task1 = {"x1": 42.9, "x2": 81.3, "distance": 1}
task2 = {"x1": 50, "x2": 49, "distance": 5}

response = agent.get_response(task1, verbose=True)
response = agent.get_response(task2, verbose=True)

## Training Code (Extra)

This section serves the purpose of showcasing the training code for the models that the agent can use for Glyph comparaison

In [ ]:
import torch
import torch.nn as nn
import src.machine_learning as ML
import os
from src.Model_BR import GlyphClassifier
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn.functional as F
import numpy as np
from torch.optim.lr_scheduler import StepLR
from pathlib import Path



config={
    "architecture": "CNN-Glyph",
    "dataset": 'data/simple-star.zip',
    "epochs": 10,
    "batch_size": 64,
    "learning_rate": 0.0005,
    "loss_fn": "MSELoss",
    "optimizer": "Adam",
    "image_resolution": (128, 128),
    "regression": True,
    "num_bins": 5,
    "rotation": 0,
    "translation": 0
}

# Getting the dataset
dataset_file = config["dataset"]
train_dataset = ML.GlyphDataset(dataset_file, resize=config["image_resolution"], split = "train",augmentation_rot=config["rotation"],augmentation_tran=config["translation"])
validation_dataset = ML.GlyphDataset(dataset_file, resize=config["image_resolution"],split = 'test',augmentation_rot=config["rotation"],augmentation_tran=config["translation"])

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=config["batch_size"], shuffle = True)
validation_loader = ML.create_loader(validation_dataset, batch_size=config["batch_size"], shuffle=False)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

bin_centers = torch.linspace(0, 100, config["num_bins"] + 1, device=device)[:-1] + 50 / config["num_bins"]

model = GlyphClassifier(resolution=config["image_resolution"], NUM_bins=config["num_bins"]).to(device)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

scheduler = StepLR(optimizer, step_size=5, gamma=0.5)  # Reduce LR by half every 5 epochs

experiment_name = f"exp-SimpleStar-{config['image_resolution'][0]}x{config['image_resolution'][1]}-{config['num_bins']}bins-BinnedRegression-withvalidation"

print(f"Experiment name: {experiment_name}")

# Initialize W&B

train_losses = []
epoch_train_losses = []
val_losses = []
global_step = 0

for epoch in range(config["epochs"]):
    model.train()
    running_loss = 0.0

    for images, values in train_loader:
        images = images.to(device)
        values = values.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.sum(probabilities * bin_centers, dim=1)
        loss = criterion(predictions, values)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_losses.append(loss.item())

        mae = F.l1_loss(predictions, values).item()

        if global_step % 100 == 0:
            print(f"Step {global_step}: Loss = {loss.item():.4f}")
        global_step += 1

    avg_train_loss = running_loss / len(train_loader)
    epoch_train_losses.append(avg_train_loss)
    print(f"Epoch {epoch+1}/{config['epochs']} - Train Loss: {avg_train_loss:.4f}")

    # Validation at the end of the epoch 
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, values in validation_loader:
            images = images.to(device)
            values = values.to(device)
            logits = model(images)
            probabilities = F.softmax(logits, dim=1)
            preds = torch.sum(probabilities * bin_centers, dim=1)
            val_loss += criterion(preds, values).item()
    
    scheduler.step()

    avg_val_loss = val_loss / len(validation_loader)
    val_losses.append(avg_val_loss)
    print(f"Epoch {epoch+1}/{config['epochs']} - Val Loss: {avg_val_loss:.4f}")



# === Extract dataset base name and generate model path ===
dataset_path = config["dataset"]
base_name = Path(dataset_path).stem  
model_filename = f"{base_name}.pt"   
model_path = os.path.join("data", model_filename)  

# === Save the model ===
torch.save(model.state_dict(), model_path)
print(f"✅ Model saved to: {model_path}")